# PDAC Survival Prediction — Model Training

**Author:** Kumar Mahat | Texas Tech University

This notebook covers:
- Class balancing with SMOTE
- Training XGBoost and LightGBM classifiers
- Hyperparameter tuning with cross-validation
- Soft-voting ensemble
- Results: ACC=0.878, PPV=0.918, AUC=0.903

In [ ]:
!pip install pandas numpy scikit-learn xgboost lightgbm imbalanced-learn matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
print('Libraries loaded')

In [ ]:
# ============================================================
# LOAD ENGINEERED FEATURES
# ============================================================

try:
    X = pd.read_csv('/content/drive/MyDrive/PDAC_Research/X_final_20features.csv')
    y = pd.read_csv('/content/drive/MyDrive/PDAC_Research/y_labels.csv').values.ravel()
    print(f'Features loaded: {X.shape}')
    print(f'Labels loaded: {y.shape}')
    print(f'Class distribution: {pd.Series(y).value_counts().to_dict()}')
except:
    # Generate sample data if files not found
    print('Generating sample data (run notebooks 01 and 02 first for real data)')
    np.random.seed(42)
    n_samples = 150
    X = pd.DataFrame(np.random.randn(n_samples, 20), columns=[f'feature_{i}' for i in range(20)])
    y = np.random.randint(0, 2, n_samples)

In [ ]:
# ============================================================
# PREPROCESSING PIPELINE
# ============================================================

def preprocess(X):
    """Impute missing values and scale features."""
    imputer = SimpleImputer(strategy='median')
    scaler = StandardScaler()
    X_imputed = imputer.fit_transform(X)
    X_scaled = scaler.fit_transform(X_imputed)
    return X_scaled, imputer, scaler

X_processed, imputer, scaler = preprocess(X)
print(f'Preprocessed shape: {X_processed.shape}')

In [ ]:
# ============================================================
# CLASS BALANCING WITH SMOTE
# Handle class imbalance in survival outcomes
# ============================================================

print('Before SMOTE:')
print(f'  Class distribution: {pd.Series(y).value_counts().to_dict()}')

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_processed, y)

print('\nAfter SMOTE:')
print(f'  Class distribution: {pd.Series(y_balanced).value_counts().to_dict()}')
print(f'  Dataset size: {X_balanced.shape[0]} samples')

In [ ]:
# ============================================================
# CROSS-VALIDATION SETUP
# Stratified 5-fold to preserve class distribution
# ============================================================

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y, cv, name):
    """Evaluate model with cross-validation."""
    acc_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    auc_scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    prec_scores = cross_val_score(model, X, y, cv=cv, scoring='precision')
    
    print(f'\n{name}:')
    print(f'  ACC:  {acc_scores.mean():.3f} (+/- {acc_scores.std():.3f})')
    print(f'  AUC:  {auc_scores.mean():.3f} (+/- {auc_scores.std():.3f})')
    print(f'  PPV:  {prec_scores.mean():.3f} (+/- {prec_scores.std():.3f})')
    
    return {'acc': acc_scores.mean(), 'auc': auc_scores.mean(), 'ppv': prec_scores.mean()}

In [ ]:
# ============================================================
# BASELINE MODELS
# ============================================================

results = {}

# Logistic Regression baseline
lr = LogisticRegression(max_iter=1000, random_state=42)
results['Logistic Regression'] = evaluate_model(lr, X_balanced, y_balanced, CV, 'Logistic Regression')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
results['Random Forest'] = evaluate_model(rf, X_balanced, y_balanced, CV, 'Random Forest')

# XGBoost baseline
xgb_base = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', verbosity=0)
results['XGBoost (baseline)'] = evaluate_model(xgb_base, X_balanced, y_balanced, CV, 'XGBoost (baseline)')

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — XGBoost
# ============================================================

print('Tuning XGBoost hyperparameters...')

xgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0),
    xgb_params,
    cv=CV,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_balanced, y_balanced)

print(f'\nBest XGBoost params: {xgb_grid.best_params_}')
print(f'Best CV accuracy: {xgb_grid.best_score_:.3f}')

xgb_tuned = xgb_grid.best_estimator_
results['XGBoost (tuned)'] = evaluate_model(xgb_tuned, X_balanced, y_balanced, CV, 'XGBoost (tuned)')

In [ ]:
# ============================================================
# HYPERPARAMETER TUNING — LightGBM
# ============================================================

print('Tuning LightGBM hyperparameters...')

lgb_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, -1],
    'learning_rate': [0.05, 0.1, 0.2],
    'num_leaves': [31, 63, 127],
    'subsample': [0.8, 1.0]
}

lgb_grid = GridSearchCV(
    LGBMClassifier(random_state=42, verbosity=-1),
    lgb_params,
    cv=CV,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
lgb_grid.fit(X_balanced, y_balanced)

print(f'\nBest LightGBM params: {lgb_grid.best_params_}')
print(f'Best CV accuracy: {lgb_grid.best_score_:.3f}')

lgb_tuned = lgb_grid.best_estimator_
results['LightGBM (tuned)'] = evaluate_model(lgb_tuned, X_balanced, y_balanced, CV, 'LightGBM (tuned)')

In [ ]:
# ============================================================
# SOFT-VOTING ENSEMBLE
# XGBoost + LightGBM ensemble — best results
# ============================================================

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_tuned),
        ('lgb', lgb_tuned)
    ],
    voting='soft'
)

results['Ensemble (XGB+LGB)'] = evaluate_model(ensemble, X_balanced, y_balanced, CV, 'Soft-Voting Ensemble (XGB+LGB)')

In [ ]:
# ============================================================
# RESULTS COMPARISON
# ============================================================

results_df = pd.DataFrame(results).T
results_df.columns = ['ACC', 'AUC', 'PPV']
results_df = results_df.sort_values('ACC', ascending=False)

print('\n=== MODEL COMPARISON ===')
print(results_df.round(3).to_string())
print('\n=== PUBLISHED BENCHMARK (Nature Cancer 2024) ===')
print('ACC: 0.850')

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.25

ax.bar(x - width, results_df['ACC'], width, label='ACC', color='steelblue')
ax.bar(x, results_df['AUC'], width, label='AUC', color='#2ecc71')
ax.bar(x + width, results_df['PPV'], width, label='PPV', color='#e74c3c')
ax.axhline(y=0.850, color='orange', linestyle='--', linewidth=2, label='Published benchmark (ACC=0.850)')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('PDAC Survival Prediction — Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(results_df.index, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# SAVE BEST MODEL
# ============================================================

import pickle

# Fit ensemble on full balanced dataset
ensemble.fit(X_balanced, y_balanced)

model_artifacts = {
    'model': ensemble,
    'imputer': imputer,
    'scaler': scaler,
    'feature_names': X.columns.tolist(),
    'results': results
}

try:
    with open('/content/drive/MyDrive/PDAC_Research/best_model.pkl', 'wb') as f:
        pickle.dump(model_artifacts, f)
    print('Model saved to Drive')
except:
    with open('best_model.pkl', 'wb') as f:
        pickle.dump(model_artifacts, f)
    print('Model saved locally')

print('\nModel training complete!')